# danvas: dynamic panels from a notebook

`serve(block=False)` starts the server without blocking, so each cell returns
and you can keep inserting panels onto the **already-open** canvas. New
components are broadcast live to connected browsers.

`insert(component, x=, y=, w=, h=)` lets you place and size each panel: pass
`x`/`y` (canvas coordinates) to position it, `w`/`h` (pixels) to size it. Omit
any of them to fall back to auto-cascade / default size.

In [ ]:
import danvas

# Starts the server in a background thread and opens the browser.
# Returns immediately so the cell finishes.
canvas = danvas.Canvas().serve(port=8000, block=False)

The canvas is now empty in your browser. Run the next cells one at a time and
watch each panel appear live.

In [ ]:
canvas.inspector(name="inspector", refresh=1.0, x=100, y=500)

In [ ]:
# Place panels explicitly with x/y (canvas coords) and size them with w/h.
servo = canvas.insert(
    danvas.Slider(name="servo_1", min=0, max=180, default=90), x=180, y=100, w=220, h=100, draggable=False, resizable=False)
status = canvas.insert(danvas.Label(name="status", value="idle", label="status box"), x=600, y=350, rotation=0)
arrow = canvas.connect(servo, status, text=f"{servo.value} times 2", dash="dashed")

@servo.on_change
def on_servo(value):
    status.update(f"{value*2}")
    arrow.update(text=f"{value} times 2")
    status.set_layout(y=canvas.servo_1.y + value)  # slider value drives the label's y position

In [ ]:
canvas.connect(status, canvas["inspector"], text="jit arrow", dash="dotted")

In [ ]:
canvas.insert(servo)

Drag the slider in the browser — `on_servo` fires here and updates the label.

Now add another panel later, with the page still open:

In [ ]:
# Inserted after serve + after the page loaded — appears live, below the slider.
mode = canvas.insert(
    danvas.Toggle(name="mode", options=["idle", "run", "stop"], default="idle"),
    x=120, y=310, w=320, h=110,
)

@mode.on_change
def on_mode(value):
    print("mode =", value)
    canvas.status.rotation = canvas.status.rotation + 1  # rotate the label when the toggle changes

In [ ]:
canvas.remove(mode)

In [ ]:
# Push a value from Python at any time.
status.update("updated from the notebook")

When you're done, shut the background server down:

In [ ]:
# canvas.stop()